# 🏠 House Price Prediction using Linear Regression

Predicts house prices based on features like **area, number of rooms, location** and more.

**Dataset:** [Kaggle House Prices — Advanced Regression Techniques](https://www.kaggle.com/competitions/house-prices-advanced-regression-techniques)

**Skills covered:** Data Preprocessing · Linear Regression · Evaluation Metrics

---

## 📦 Step 1: Install & Import Libraries

In [ ]:
!pip install kaggle -q

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.impute import SimpleImputer

plt.style.use('ggplot')
print('✅ All libraries imported successfully!')

## 📥 Step 2: Download Dataset from Kaggle

> Upload your `kaggle.json` to Colab before running.
> Get it from: kaggle.com → Profile → Settings → API → Create New Token

In [ ]:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

# Download House Prices dataset
!kaggle competitions download -c house-prices-advanced-regression-techniques

from zipfile import ZipFile
with ZipFile('house-prices-advanced-regression-techniques.zip', 'r') as z:
    z.extractall()

print('✅ Dataset downloaded!')
!ls *.csv

In [ ]:
# Load train data
df = pd.read_csv('train.csv')
print(f'Dataset shape: {df.shape}')
print(f'Rows: {df.shape[0]} | Features: {df.shape[1]}')
df.head()

## 🔍 Step 3: Exploratory Data Analysis (EDA)

In [ ]:
print('Dataset Info:')
print(f'Shape     : {df.shape}')
print(f'Duplicates: {df.duplicated().sum()}')
print(f'\nTarget variable (SalePrice) stats:')
print(df['SalePrice'].describe())

In [ ]:
# Sale Price Distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df['SalePrice'], bins=40, color='steelblue', edgecolor='black')
axes[0].set_title('SalePrice Distribution', fontsize=13)
axes[0].set_xlabel('Sale Price ($)')
axes[0].set_ylabel('Count')

axes[1].hist(np.log1p(df['SalePrice']), bins=40, color='darkorange', edgecolor='black')
axes[1].set_title('Log(SalePrice) Distribution — More Normal', fontsize=13)
axes[1].set_xlabel('Log Sale Price')
axes[1].set_ylabel('Count')

plt.tight_layout()
plt.show()

In [ ]:
# Top correlated features with SalePrice
numeric_df = df.select_dtypes(include=[np.number])
correlations = numeric_df.corr()['SalePrice'].sort_values(ascending=False)

print('Top 15 features correlated with SalePrice:')
print(correlations.head(16))  # 16 includes SalePrice itself

In [ ]:
# Correlation heatmap — top 12 features
top_features = correlations.head(12).index.tolist()
plt.figure(figsize=(12, 8))
sns.heatmap(numeric_df[top_features].corr(), annot=True, fmt='.2f',
            cmap='coolwarm', square=True, linewidths=0.5)
plt.title('Correlation Heatmap — Top Features', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Key feature scatter plots vs SalePrice
key_features = ['GrLivArea', 'TotalBsmtSF', 'GarageArea', 'OverallQual']
fig, axes = plt.subplots(1, 4, figsize=(18, 4))

for ax, feat in zip(axes, key_features):
    ax.scatter(df[feat], df['SalePrice'], alpha=0.4, color='steelblue', s=10)
    ax.set_xlabel(feat)
    ax.set_ylabel('SalePrice')
    ax.set_title(f'{feat} vs SalePrice')

plt.tight_layout()
plt.show()

In [ ]:
# Missing value analysis
missing = df.isnull().sum()
missing = missing[missing > 0].sort_values(ascending=False)
missing_pct = (missing / len(df) * 100).round(2)

missing_df = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})
print(f'Columns with missing values: {len(missing_df)}')
print(missing_df.head(20))

In [ ]:
# Visualize top 15 missing columns
plt.figure(figsize=(12, 5))
sns.barplot(x=missing_pct.head(15).values, y=missing_pct.head(15).index, palette='Reds_r')
plt.title('Top 15 Columns with Missing Data (%)', fontsize=13)
plt.xlabel('Missing %')
plt.tight_layout()
plt.show()

## 🧹 Step 4: Data Preprocessing

In [ ]:
# Select top numerical features based on correlation
FEATURES = [
    'GrLivArea',      # Above ground living area (sq ft)
    'TotalBsmtSF',    # Total basement area
    'GarageArea',     # Garage area
    'FullBath',       # Full bathrooms
    'TotRmsAbvGrd',   # Total rooms above ground
    'YearBuilt',      # Year built
    'YearRemodAdd',   # Year remodeled
    'OverallQual',    # Overall quality (1–10)
    'OverallCond',    # Overall condition
    '1stFlrSF',       # First floor sq ft
    'LotArea',        # Lot size
    'BedroomAbvGr',   # Bedrooms above ground
]
TARGET = 'SalePrice'

df_model = df[FEATURES + [TARGET]].copy()
print(f'Selected features: {len(FEATURES)}')
print(f'Dataset shape: {df_model.shape}')

In [ ]:
# Handle missing values — fill numeric with median
imputer = SimpleImputer(strategy='median')
df_model[FEATURES] = imputer.fit_transform(df_model[FEATURES])

print(f'Missing values after imputation: {df_model.isnull().sum().sum()}')

# Remove outliers — drop extreme GrLivArea outliers (Kaggle recommendation)
df_model = df_model[~((df_model['GrLivArea'] > 4000) & (df_model[TARGET] < 200000))]
print(f'Shape after outlier removal: {df_model.shape}')

# Log-transform target for better linearity
df_model['LogSalePrice'] = np.log1p(df_model[TARGET])
print('✅ Preprocessing complete!')

## ✂️ Step 5: Train / Test Split & Feature Scaling

In [ ]:
X = df_model[FEATURES]
y = df_model['LogSalePrice']   # predicting log price for better regression fit

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Standard Scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

print(f'Training samples : {X_train.shape[0]}')
print(f'Test samples     : {X_test.shape[0]}')

## 📈 Step 6: Train Linear Regression Model

In [ ]:
# Train Linear Regression
lr = LinearRegression()
lr.fit(X_train_scaled, y_train)

# Predictions (convert back from log scale)
y_pred_log  = lr.predict(X_test_scaled)
y_pred      = np.expm1(y_pred_log)
y_test_actual = np.expm1(y_test)

print('✅ Linear Regression model trained!')

## 📊 Step 7: Evaluate the Model

In [ ]:
def evaluate_model(name, y_true, y_pred):
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2   = r2_score(y_true, y_pred)
    print(f'─── {name} ───')
    print(f'MAE  (Mean Absolute Error) : ${mae:,.0f}')
    print(f'RMSE (Root Mean Sq. Error) : ${rmse:,.0f}')
    print(f'R²   (Accuracy Score)      : {r2:.4f} ({r2*100:.2f}%)')
    print()
    return {'Model': name, 'MAE': mae, 'RMSE': rmse, 'R2': r2}

results = []
results.append(evaluate_model('Linear Regression', y_test_actual, y_pred))

In [ ]:
# Actual vs Predicted plot
plt.figure(figsize=(8, 6))
plt.scatter(y_test_actual, y_pred, alpha=0.5, color='steelblue', s=20)
plt.plot([y_test_actual.min(), y_test_actual.max()],
         [y_test_actual.min(), y_test_actual.max()],
         'r--', linewidth=2, label='Perfect Prediction')
plt.xlabel('Actual Sale Price ($)')
plt.ylabel('Predicted Sale Price ($)')
plt.title('Linear Regression — Actual vs Predicted', fontsize=13)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Residuals plot
residuals = y_test_actual - y_pred

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(y_pred, residuals, alpha=0.5, color='darkorange', s=15)
axes[0].axhline(0, color='red', linestyle='--')
axes[0].set_xlabel('Predicted Price ($)')
axes[0].set_ylabel('Residual ($)')
axes[0].set_title('Residuals vs Predicted', fontsize=13)

axes[1].hist(residuals, bins=40, color='steelblue', edgecolor='black')
axes[1].set_title('Residual Distribution', fontsize=13)
axes[1].set_xlabel('Residual ($)')
axes[1].set_ylabel('Count')

plt.tight_layout()
plt.show()

In [ ]:
# Feature importance (coefficients)
coef_df = pd.DataFrame({
    'Feature': FEATURES,
    'Coefficient': lr.coef_
}).sort_values('Coefficient', key=abs, ascending=False)

plt.figure(figsize=(10, 5))
colors = ['#2ecc71' if c > 0 else '#e74c3c' for c in coef_df['Coefficient']]
plt.barh(coef_df['Feature'], coef_df['Coefficient'], color=colors, edgecolor='black')
plt.axvline(0, color='black', linewidth=0.8)
plt.title('Feature Coefficients — Linear Regression', fontsize=13)
plt.xlabel('Coefficient Value')
plt.tight_layout()
plt.show()

print(coef_df.to_string(index=False))

## 🔁 Step 8: Compare with Ridge & Lasso Regression

In [ ]:
# Ridge Regression
ridge = Ridge(alpha=10)
ridge.fit(X_train_scaled, y_train)
ridge_pred = np.expm1(ridge.predict(X_test_scaled))
results.append(evaluate_model('Ridge Regression', y_test_actual, ridge_pred))

# Lasso Regression
lasso = Lasso(alpha=0.001)
lasso.fit(X_train_scaled, y_train)
lasso_pred = np.expm1(lasso.predict(X_test_scaled))
results.append(evaluate_model('Lasso Regression', y_test_actual, lasso_pred))

In [ ]:
# Model comparison table
results_df = pd.DataFrame(results)
results_df['MAE']  = results_df['MAE'].apply(lambda x: f'${x:,.0f}')
results_df['RMSE'] = results_df['RMSE'].apply(lambda x: f'${x:,.0f}')
results_df['R2']   = results_df['R2'].apply(lambda x: f'{x:.4f}')
print('Model Comparison:')
print(results_df.to_string(index=False))

In [ ]:
# Cross-validation score for Linear Regression
cv_scores = cross_val_score(lr, X_train_scaled, y_train, cv=5, scoring='r2')
print(f'Cross-Validation R² Scores : {cv_scores.round(4)}')
print(f'Mean CV R²                 : {cv_scores.mean():.4f}')
print(f'Std CV R²                  : {cv_scores.std():.4f}')

## 🔮 Step 9: Predict on a Custom House

In [ ]:
def predict_house_price(features_dict):
    """
    Predict house price given a dictionary of feature values.
    Features: GrLivArea, TotalBsmtSF, GarageArea, FullBath,
              TotRmsAbvGrd, YearBuilt, YearRemodAdd, OverallQual,
              OverallCond, 1stFlrSF, LotArea, BedroomAbvGr
    """
    input_df = pd.DataFrame([features_dict])[FEATURES]
    input_scaled = scaler.transform(input_df)
    log_price = lr.predict(input_scaled)[0]
    price = np.expm1(log_price)
    print(f'🏠 Estimated House Price: ${price:,.0f}')
    return price

# ── Try your own house specs below! ──────────────────────────────
predict_house_price({
    'GrLivArea'    : 1800,   # sq ft living area
    'TotalBsmtSF'  : 900,    # basement sq ft
    'GarageArea'   : 400,    # garage sq ft
    'FullBath'     : 2,      # full bathrooms
    'TotRmsAbvGrd' : 7,      # total rooms
    'YearBuilt'    : 2005,   # year built
    'YearRemodAdd' : 2010,   # year remodeled
    'OverallQual'  : 7,      # quality (1-10)
    'OverallCond'  : 6,      # condition (1-10)
    '1stFlrSF'     : 1000,   # 1st floor sq ft
    'LotArea'      : 8500,   # lot size sq ft
    'BedroomAbvGr' : 3,      # bedrooms
})

## 💾 Step 10: Save Results

In [ ]:
import pickle

# Save model and scaler
with open('house_price_lr_model.pkl', 'wb') as f:
    pickle.dump(lr, f)
with open('house_price_scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

# Save predictions to CSV
output = X_test.copy()
output['ActualPrice']    = y_test_actual.values
output['PredictedPrice'] = y_pred
output['Error']          = output['ActualPrice'] - output['PredictedPrice']
output.to_csv('house_price_predictions.csv', index=False)

print('✅ Model saved  → house_price_lr_model.pkl')
print('✅ Scaler saved → house_price_scaler.pkl')
print('✅ Results saved → house_price_predictions.csv')

---
## 📋 Summary

| Step | What was done |
|------|---------------|
| EDA | Distribution, correlations, missing values |
| Preprocessing | Median imputation, outlier removal, log transform |
| Features | 12 key numerical features selected by correlation |
| Model | Linear Regression + Ridge + Lasso compared |
| Metrics | MAE, RMSE, R² + 5-fold Cross Validation |
| Output | Saved model, scaler, and predictions CSV |

**Key Insight:** `OverallQual`, `GrLivArea`, and `TotalBsmtSF` are the strongest predictors of house price.